# The Full Constitutional Loop [Step 04.03]

> **MLCourse - Agentic AI - Agent Patterns**

Now we run it end to end:

```
   draft
     |
     v
   +-----------------------------------------------+
   |  for each principle: does the draft break it? |  <- CRITIQUE
   +-----------------------------------------------+
     |  violations found?
     |         yes                    no
     v                                 \
   revise against ALL violations        -> done
     |                                       ^
     +---------------------------------------+
        (bounded: at most MAX_ROUNDS)
```

### What you'll learn

- Per-principle critique, and why it beats one critique of everything.
- The revision step, and the instruction that stops it rewriting the whole email.
- **Measured** violations per round, plus what it cost in calls and tokens.
- Convergence, oscillation, and how to detect the difference.

### Why it matters

This is the pattern behind the "harmlessness" training of several frontier
models, and it works at inference time too. Knowing where it converges - and
where it just spends money - is the difference between using it and cargo-culting
it.

### Prerequisites

- [02_writing_a_constitution](02_writing_a_constitution.ipynb)

### Setup: environment, model, token counting, rate-limit-aware call helper


In [ ]:
import os                              # environment variables
import time                            # timing and pacing
import json                            # pretty-printing structured context
from pathlib import Path               # locating the track root
from dotenv import load_dotenv         # reads KEY=value pairs from .env

# Walk UP from the notebook folder until we hit the repo root, then load the
# (gitignored) .env that lives inside 03_agentic_ai. Note the extra path
# segment: the walk-up lands on the REPO ROOT, not on the track folder.
TRACK = Path.cwd()
while not (TRACK / "03_agentic_ai").exists() and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / "03_agentic_ai" / ".env")

GROQ_MODEL = "qwen/qwen3.8-27b"        # hosted, fast, generous free tier
# Local alternative (documented, not used here): Ollama `llama3.1:8b` via
# `from langchain_ollama import ChatOllama`. OpenAI is never used in this course.

from langchain_groq import ChatGroq


def make_llm(temperature: float = 0.0, max_tokens: int = 300, **kw):
    """One place that constructs the chat model, so every notebook is identical."""
    return ChatGroq(model=GROQ_MODEL, temperature=temperature,
                    max_tokens=max_tokens, **kw)


# --- Token counting -----------------------------------------------------------
# Two different numbers, and it matters which one you are looking at:
#   * approx_tokens(): a LOCAL estimate using tiktoken's cl100k_base. It is not
#     the model's own tokenizer, so treat it as "within ~10%", good for
#     budgeting BEFORE you send a request.
#   * usage_metadata on the response: the provider's EXACT count. Ground truth,
#     but only available AFTER you have already paid for the call.
import tiktoken

_ENC = tiktoken.get_encoding("cl100k_base")


def approx_tokens(text) -> int:
    """Approximate token count for a string (or anything str()-able)."""
    return len(_ENC.encode(str(text)))


# --- Rate-limit-aware calling --------------------------------------------------
# The Groq free tier allows 8000 tokens per minute. Several notebooks here make
# many small calls in a loop, so we self-pace well under the ceiling and retry
# with exponential backoff if we are throttled anyway.

TPM_BUDGET = 3500                       # deliberately conservative
_WINDOW = []                            # [(timestamp, tokens), ...]
USAGE = {"calls": 0, "in": 0, "out": 0, "seconds": 0.0}


def _pace(cost: int):
    """Sleep just enough that our rolling 60s token usage stays under budget."""
    now = time.time()
    while True:
        recent = [(t, n) for (t, n) in _WINDOW if now - t < 60]
        _WINDOW[:] = recent
        if sum(n for _, n in recent) + cost <= TPM_BUDGET or not recent:
            return
        time.sleep(min(5.0, 60 - (now - recent[0][0]) + 0.5))
        now = time.time()


def chat(messages, llm=None, temperature=0.0, max_tokens=300, retries=5):
    """Send `messages`, return the AIMessage. Paces, retries, and meters usage.

    `messages` is a list of (role, content) tuples or LangChain message objects.
    """
    llm = llm or make_llm(temperature=temperature, max_tokens=max_tokens)
    est = approx_tokens(messages) + max_tokens
    delay = 4.0
    for attempt in range(retries):
        _pace(est)
        t0 = time.time()
        try:
            out = llm.invoke(messages)
        except Exception as exc:
            if "rate_limit" in str(exc) or "429" in str(exc):
                time.sleep(delay)
                delay = min(delay * 2, 45)
                continue
            raise
        u = out.usage_metadata or {}
        _WINDOW.append((time.time(), u.get("total_tokens", est)))
        USAGE["calls"] += 1
        USAGE["in"] += u.get("input_tokens", 0)
        USAGE["out"] += u.get("output_tokens", 0)
        USAGE["seconds"] += time.time() - t0
        return out
    raise RuntimeError("still rate limited after %d attempts" % retries)


def ask(prompt: str, system: str = None, **kw) -> str:
    """Convenience wrapper: one user turn in, plain text out."""
    msgs = ([("system", system)] if system else []) + [("user", prompt)]
    return chat(msgs, **kw).content.strip()


print("model:", GROQ_MODEL)
print("key loaded:", bool(os.getenv("GROQ_API_KEY")))
print("tokenizer:", "cl100k_base (approximation)")


### The shared task


In [ ]:
# One drafting task, used by every notebook in this module so the comparisons are
# apples to apples. It is chosen because it has MANY ways to go subtly wrong -
# which is exactly the situation principles are for.

SITUATION = """A customer, Elena Duarte, has emailed angrily. Her order NW-10261
(SKU TH-275-BLK, supplied by Meridian Components GmbH) was cancelled by us
without warning because the frame size was discontinued. She has been waiting
eleven days. Our records show she was charged and NOT yet refunded; finance says
the refund will clear in 3-5 business days but has occasionally taken longer.
She has asked for compensation. Our policy allows a goodwill voucher of up to
15 EUR, which requires a supervisor's approval that has not yet been given."""

DRAFT_INSTRUCTION = ("Write the reply we should send to Elena. Write only the "
                     "email body, no subject line and no notes.")

print(SITUATION)


### Deterministic checks


In [ ]:
# Not every principle can be checked by code, but several can - and the ones that
# can are worth far more than an LLM's opinion, because they cannot be argued
# with. We use them to MEASURE whether critique-and-revise actually changed
# anything, rather than trusting the model's report of its own improvement.

import re

INTERNAL_TERMS = ["th-275", "meridian", "supplier", "sku", "finance says",
                  "our records show"]
OVERPROMISE = ["guarantee", "guaranteed", "i promise", "we promise", "definitely will",
               "will certainly", "rest assured that you will", "immediately refund"]
EMPATHY = ["sorry", "apolog", "understand", "frustrat", "disappoint", "regret"]


def check(text):
    """Return a dict of objective observations about a draft."""
    low = text.lower()
    words = len(re.findall(r"\b[\w'-]+\b", text))
    return {
        "words": words,
        "over_150_words": words > 150,
        "internal_leaks": [t for t in INTERNAL_TERMS if t in low],
        "overpromises": [t for t in OVERPROMISE if t in low],
        "has_empathy": any(t in low for t in EMPATHY),
        "mentions_voucher": "voucher" in low or "15 eur" in low or "15EUR".lower() in low,
        "states_refund_window": bool(re.search(r"3\s*[-to]+\s*5\s*business days", low)),
    }


def violations(text):
    """Count objective violations. Lower is better."""
    c = check(text)
    n = 0
    n += len(c["internal_leaks"])
    n += len(c["overpromises"])
    n += int(c["over_150_words"])
    n += int(not c["has_empathy"])
    return n


def report(text, label):
    c = check(text)
    print("%-22s words=%-4d leaks=%-22s overpromise=%-18s empathy=%-5s -> %d violations"
          % (label, c["words"], ",".join(c["internal_leaks"]) or "none",
             ",".join(c["overpromises"]) or "none", c["has_empathy"], violations(text)))
    return c


In [4]:
# The constitution from notebook 02, restated so this notebook stands alone.
CONSTITUTION = [
    dict(id="P1", name="No internal information",
         rule="The reply must not disclose supplier names, SKU or part codes, "
              "internal system names, or what internal departments have said.",
         fix="Remove the internal detail entirely."),
    dict(id="P2", name="No uncontrollable commitments",
         rule="The reply must not promise a specific date, guarantee an outcome, "
              "or use words like 'guarantee', 'definitely' or 'I promise' about "
              "anything the company does not directly control.",
         fix="Describe the company's normal timescale as typical, not promised."),
    dict(id="P3", name="No unapproved compensation",
         rule="The reply must not offer, imply or quote a compensation amount "
              "unless approval has already been given.",
         fix="Say a goodwill gesture is being reviewed, without an amount."),
    dict(id="P4", name="Acknowledge the impact",
         rule="The reply must explicitly acknowledge the customer's frustration "
              "and the time they have waited, before explaining anything.",
         fix="Open with a direct acknowledgement of the delay and its effect."),
    dict(id="P5", name="Give one concrete next step",
         rule="The reply must tell the customer exactly one thing that will "
              "happen next, and who will do it.",
         fix="Add one sentence naming the next action and its owner."),
    dict(id="P6", name="Brevity",
         rule="The reply must be under 150 words.",
         fix="Cut explanation, not acknowledgement."),
]

PRIORITY = ("Safety principles (P1, P2, P3) outrank everything. Then P4, then P5, "
            "then P6. Never violate a safety principle to satisfy a lower one.")

print("%d principles" % len(CONSTITUTION))

6 principles


### 1. Per-principle critique

One call per principle, not one call for all six. It costs more, and it is worth
it for a specific reason:

> A single "check all six" call reliably finds two or three violations and stops.
> Attention is finite, and the model satisfices.

Six focused calls each have one job. If cost matters more than recall, batch the
low-priority principles together and keep the safety ones separate - never the
other way round.

In [5]:
CRITIC = """You are auditing a customer service reply against ONE principle.

PRINCIPLE {pid} ({name}): {rule}

Answer in exactly this format and nothing else:
VIOLATION: <quote the exact offending text from the reply>
or
OK

Be strict but do not invent violations. If the reply does not break THIS
principle, answer OK even if something else about it is imperfect.

REPLY:
{text}"""


def critique(text):
    """Return a list of (principle, quoted offending text) for every violation."""
    found = []
    for p in CONSTITUTION:
        out = chat([("user", CRITIC.format(pid=p["id"], name=p["name"],
                                           rule=p["rule"], text=text))],
                   temperature=0.0, max_tokens=110).content.strip()
        if out.upper().startswith("VIOLATION"):
            found.append((p, out.split(":", 1)[1].strip() if ":" in out else out))
    return found

### 2. Revision

Two instructions do almost all the work here:

- **"Change nothing else."** Without it, the model rewrites the entire email
  every round, which destroys anything already good and makes the loop
  non-convergent.
- **The `fix` for each violated principle.** The model is told the remedy, not
  just the complaint - which is why notebook 02 insisted every rule carry one.

In [6]:
REVISE = """Revise the reply below so that every listed violation is fixed.

{priority}

Change nothing else. Keep the parts that were not flagged exactly as they are.
Output only the revised email body, with no preamble and no notes.

VIOLATIONS TO FIX:
{violations}

REPLY TO REVISE:
{text}"""


def revise(text, found):
    block = "\n".join("- [%s %s] offending text: %s\n  HOW TO FIX: %s"
                      % (p["id"], p["name"], quote[:90], p["fix"])
                      for p, quote in found)
    return chat([("user", REVISE.format(priority=PRIORITY, violations=block, text=text))],
                temperature=0.2, max_tokens=420).content.strip()

In [7]:
MAX_ROUNDS = 3

draft = chat([("user", SITUATION + "\n\n" + DRAFT_INSTRUCTION)],
             temperature=0.3, max_tokens=420).content.strip()

print("ROUND 0 - initial draft\n")
print(draft)
print()
_ = report(draft, "round 0")

ROUND 0 - initial draft

Dear Elena,

I am writing to sincerely apologize for the significant frustration and inconvenience caused by the cancellation of your order NW-10261. I understand that you have been waiting for eleven days without prior notice regarding the discontinuation of the frame size for SKU TH-275-BLK, and I deeply regret that we failed to communicate this change to you promptly.

I have personally reviewed your account and confirmed that while you were charged for this item, the refund has not yet been processed. I have escalated this issue with our finance team to ensure your refund is prioritized. While our standard processing time is 3-5 business days, I acknowledge that this can occasionally take longer. I will monitor this transaction closely to ensure it clears as quickly as possible and will keep you updated if there are any delays.

Regarding your request for compensation, I understand how this experience has affected your trust in our service. I am currently s

In [8]:
history = [("round 0", draft)]
calls_at_start = USAGE["calls"]
current = draft

for r in range(1, MAX_ROUNDS + 1):
    found = critique(current)
    print("\nROUND %d - critique found %d violation(s)" % (r, len(found)))
    for p, quote in found:
        print("  %s %-28s %s" % (p["id"], p["name"], quote[:56]))

    if not found:
        print("  -> constitution satisfied, stopping.")
        break

    current = revise(current, found)
    history.append(("round %d" % r, current))
    print()
    _ = report(current, "round %d" % r)

print("\nFINAL REPLY:\n")
print(current)


ROUND 1 - critique found 4 violation(s)
  P1 No internal information      SKU TH-275-BLK
  P3 No unapproved compensation   I am currently seeking supervisor approval to issue a go
  P5 Give one concrete next step  I have escalated this issue with our finance team to ens
  P6 Brevity                      Dear Elena,

I am writing to sincerely apologize for the



round 1                words=146  leaks=none                   overpromise=none               empathy=True  -> 0 violations



ROUND 2 - critique found 2 violation(s)
  P5 Give one concrete next step  A goodwill gesture is currently being reviewed, and I wi
  P6 Brevity                      Dear Elena,

I sincerely apologize for the frustration a



round 2                words=57   leaks=none                   overpromise=none               empathy=True  -> 0 violations



ROUND 3 - critique found 3 violation(s)
  P2 No uncontrollable commitments I am personally reviewing the goodwill gesture and will 
  P4 Acknowledge the impact       Thank you for your patience.
  P5 Give one concrete next step  Your refund is being processed now; you will receive a c



round 3                words=67   leaks=none                   overpromise=none               empathy=True  -> 0 violations

FINAL REPLY:

Dear Elena,

I apologize for the cancellation of order NW-10261 and the lack of prior notice, which I understand has caused significant inconvenience.

Your refund is being processed now; you will receive a confirmation email within 3-5 business days. The refund team is currently executing this transaction.

The goodwill gesture is under review, and we typically provide a decision within one business day.

Sincerely,

Customer Support Team


### 3. Did it actually improve? Measure.

The critique is the model's opinion. The `check()` function is not. Compare every
round on the objective checks.

In [9]:
print("%-10s %6s %8s %12s %8s %10s" % ("round", "words", "leaks", "overpromises",
                                       "empathy", "violations"))
print("-" * 62)
for label, text in history:
    c = check(text)
    print("%-10s %6d %8d %12d %8s %10d"
          % (label, c["words"], len(c["internal_leaks"]), len(c["overpromises"]),
             "yes" if c["has_empathy"] else "NO", violations(text)))

v_first, v_last = violations(history[0][1]), violations(history[-1][1])
print()
print("MEASURED, this run, %s:" % GROQ_MODEL)
print("  objective violations : %d -> %d  (%+d)" % (v_first, v_last, v_last - v_first))
print("  rounds run           : %d" % (len(history) - 1))
print("  LLM calls spent      : %d (%d critiques + %d revisions)"
      % (USAGE["calls"] - calls_at_start,
         len(CONSTITUTION) * (len(history) - 1 + (1 if v_last == 0 else 0)),
         len(history) - 1))
print("  total tokens         : %d in, %d out" % (USAGE["in"], USAGE["out"]))

round       words    leaks overpromises  empathy violations
--------------------------------------------------------------
round 0       220        2            0      yes          3
round 1       146        0            0      yes          0
round 2        57        0            0      yes          0
round 3        67        0            0      yes          0

MEASURED, this run, qwen/qwen3.8-27b:
  objective violations : 3 -> 0  (-3)
  rounds run           : 3
  LLM calls spent      : 21 (24 critiques + 3 revisions)
  total tokens         : 6737 in, 1050 out


In [10]:
if v_last < v_first:
    print("The loop removed %d objective violation(s)." % (v_first - v_last))
    print("Note WHICH ones: the checkable rules (leaks, overpromises, length) are")
    print("exactly the ones code can verify. The unverifiable principles - P4, P5 -")
    print("were also critiqued, but we have only the model's word that they improved.")
elif v_last == v_first == 0:
    print("The draft satisfied every objective check from the start.")
    print("Reported honestly: on this run the loop cost extra calls and changed")
    print("nothing measurable. That happens when the base model is already careful,")
    print("and it is an argument for critiquing SELECTIVELY rather than always.")
else:
    print("Objective violations did not fall. Check the per-round table: a rising")
    print("word count with flat violations is the classic sign that revision is")
    print("rewriting rather than fixing.")

The loop removed 3 objective violation(s).
Note WHICH ones: the checkable rules (leaks, overpromises, length) are
exactly the ones code can verify. The unverifiable principles - P4, P5 -
were also critiqued, but we have only the model's word that they improved.


### 4. Convergence and oscillation

Three ways a critique loop can end:

| Ending | What it looks like | What to do |
|---|---|---|
| **Convergence** | Violations fall to zero | Stop. This is the good case. |
| **Oscillation** | Round 2 re-introduces what round 1 fixed | Your principles overlap - merge them |
| **Churn** | Text changes every round, violations do not | Your critique is not grounded - add a checker |

Detecting oscillation is cheap: record which principle fired each round and look
for a principle that fires, stops, and fires again.

### Re-critique every round we produced, and look for a principle that comes back.


In [ ]:
def objective_fires(text):
    """Which principles does the DETERMINISTIC checker say are broken?"""
    ids = set()
    c = check(text)
    if c["internal_leaks"]:
        ids.add("P1")
    if c["overpromises"]:
        ids.add("P2")
    if c["mentions_voucher"]:
        ids.add("P3")
    if not c["has_empathy"]:
        ids.add("P4")
    if c["over_150_words"]:
        ids.add("P6")
    return ids


fired_by_round = [(label, objective_fires(text)) for label, text in history]

for label, ids in fired_by_round:
    print("%-10s %s" % (label, ", ".join(sorted(ids)) or "(clean on objective checks)"))

# Oscillation = a principle that fired, then cleared, then fired again.
oscillating = set()
for p in CONSTITUTION:
    seq = [p["id"] in ids for _, ids in fired_by_round]
    for i in range(len(seq) - 2):
        if seq[i] and not seq[i + 1] and any(seq[i + 2:]):
            oscillating.add(p["id"])

print()
print("oscillating principles:", sorted(oscillating) or "none detected")
print("(A principle that fires, clears, then fires again means two principles are")
print(" fighting: fixing one re-introduces the other. Merge or narrow them.)")


> Note that the detector above uses the **objective** checks, not the LLM
> critique. Using the critic to detect the critic's own instability would be
> circular. Wherever you have a deterministic check, prefer it - the LLM critique
> is for the principles code cannot express.

### 5. When to run the loop at all

It is not free: six critique calls plus one revision per round. Run it
selectively.

| Run the constitution when | Skip it when |
|---|---|
| The output is customer-facing or published | Internal, low-stakes output |
| A mistake is expensive (legal, medical, financial) | A human reviews everything anyway |
| A cheap pre-filter flagged the draft | The pre-filter says it is clean |
| You need an auditable record of the check | Latency is the binding constraint |

Row three is the practical pattern: run the deterministic `check()` on every
draft, and invoke the full LLM constitution only on drafts it flags. Code is free
and instant; the constitution is neither.

In [12]:
def gated_review(text):
    """Run the expensive constitutional loop only if a free check flags something."""
    c = check(text)
    cheap_flags = (c["internal_leaks"] or c["overpromises"] or c["over_150_words"]
                   or not c["has_empathy"])
    if not cheap_flags:
        return text, 0, "passed the free check - no LLM critique needed"
    found = critique(text)
    if not found:
        return text, len(CONSTITUTION), "LLM critique found nothing"
    return revise(text, found), len(CONSTITUTION) + 1, "revised for %d violation(s)" % len(found)


out, spent, why = gated_review(current)
print("gate result: %s  (LLM calls spent: %d)" % (why, spent))

gate result: passed the free check - no LLM critique needed  (LLM calls spent: 0)


### 6. Pitfalls

- **One critique call for all principles.** The model finds two and stops.
- **Omitting "change nothing else".** The revision rewrites everything and the
  loop never converges.
- **Unbounded rounds.** Cap at two or three; the gains after that are nil.
- **Believing the critique.** Grade with code wherever code can express the rule.
- **Running it on every draft.** Gate it behind a free deterministic check.

### Recap

| Idea | Takeaway |
|---|---|
| Critique per principle | Six focused calls beat one call finding two problems |
| Rule + fix + "change nothing else" | The three ingredients of a stable revision |
| Measure objectively | The critique's self-report is not evidence |
| Watch for oscillation | A principle that fires, clears, and fires again means overlap |
| Gate the loop | Free checks first; spend LLM calls only on flagged drafts |

**Next:** [04_constitutional_vs_reflexion](04_constitutional_vs_reflexion.ipynb) -
the same loop shape, a completely different source of truth.